In [1]:
import requests
import pandas as pd
import time
import re
import os

os.chdir(r'A:\Coding\PycharmProjects\cryptoguard')

HEADERS = {'User-Agent': 'Mozilla/5.0 (academic research scraper)'}

def scrape_subreddit_json(subreddit, category='top', limit=100, time_filter='all'):
    url = f'https://www.reddit.com/r/{subreddit}/{category}.json?limit={limit}&t={time_filter}'
    try:
        r = requests.get(url, headers=HEADERS, timeout=10)
        if r.status_code == 200:
            return r.json()['data']['children']
        else:
            print(f"Status {r.status_code} for r/{subreddit}")
            return []
    except Exception as e:
        print(f"Error scraping r/{subreddit}: {e}")
        return []

def extract_scam_text(post):
    title = post['data'].get('title', '')
    body  = post['data'].get('selftext', '')
    combined = f"{title} {body}".strip()
    #filter out removed/deleted posts
    if body in ['[removed]', '[deleted]', '']:
        return title if len(title) > 30 else None
    if len(combined) < 30:
        return None
    return combined[:1000]  # cap at 1000 chars

#subreddits to scrape for phishing examples
PHISHING_SUBS = [
    'CryptoScams',
    'BitcoinScams',
    'Scams',
    'CryptoCurrency',
    'ethereum',
]

#subreddits for legitimate crypto communications
LEGIT_SUBS = [
    'CryptoCurrency',
    'Bitcoin',
    'ethereum',
    'defi',
    'binance',
]

phishing_texts = []
legit_texts = []

#scrape phishing examples
for sub in PHISHING_SUBS[:3]:  #CryptoScams, BitcoinScams, Scams
    posts = scrape_subreddit_json(sub, 'top', 100, 'all')
    for post in posts:
        text = extract_scam_text(post)
        if text and any(kw in text.lower() for kw in [
            'scam', 'phish', 'fraud', 'hack', 'stolen', 'lost',
            'seed phrase', 'wallet', 'crypto', 'bitcoin', 'eth',
            'airdrop', 'giveaway', 'fake', 'impersonat'
        ]):
            phishing_texts.append(text)
    print(f"r/{sub}: {len(posts)} posts scraped")
    time.sleep(2)

print(f"\nTotal phishing candidates: {len(phishing_texts)}")

#scrape legitimate crypto communications
print("\nScraping legitimate crypto subreddits...")
for sub in LEGIT_SUBS[1:]:  # Bitcoin, ethereum, defi, binance
    posts = scrape_subreddit_json(sub, 'top', 100, 'all')
    for post in posts:
        text = extract_scam_text(post)
        if text and not any(kw in text.lower() for kw in [
            'scam', 'phish', 'fraud', 'hack', 'stolen', 'lost',
            'seed phrase', 'giveaway', 'fake', 'impersonat', 'warning'
        ]):
            legit_texts.append(text)
    print(f"r/{sub}: {len(posts)} posts scraped")
    time.sleep(2)

print(f"\nTotal legitimate candidates: {len(legit_texts)}")

#preview samples
print("\nSample phishing texts")
for t in phishing_texts[:3]:
    print(f"\n{t[:200]}")

print("\nSample legitimate texts")
for t in legit_texts[:3]:
    print(f"\n{t[:200]}")

Scraping phishing subreddits...
r/CryptoScams: 100 posts scraped
r/BitcoinScams: 29 posts scraped
r/Scams: 100 posts scraped

Total phishing candidates: 183

Scraping legitimate crypto subreddits...
r/Bitcoin: 100 posts scraped
r/ethereum: 100 posts scraped
r/defi: 100 posts scraped
r/binance: 100 posts scraped

Total legitimate candidates: 259

=== Sample phishing texts ===

My dad lost 250k+  My father was scammed for 250k

Context:

Timeframe: Past 2 months
 

My father is in his mid 50’s and just lost 250k+ in “investments” from a company known as Berge. They have a we
--------------------------------------------------

“MemeCoin Whale Pumps” is a scam There is a group managed by an Instagram profile called @SpartaCrypto_

They promote a Telegram group where it suppose they share with users the next coin so everyone 
--------------------------------------------------

Rollblock Curious to know people's thoughts on Rollblock...is this presale a scam?
--------------------------------

In [2]:
#check how many phishing posts contain quoted scam messages
quoted = []
for text in phishing_texts:
    #look for posts that paste actual message content
    if any(indicator in text.lower() for indicator in [
        'received this', 'got this message', 'they said',
        'message said', 'email said', 'they sent',
        'copy paste', 'exact message', 'verbatim',
        '"', 'dm said', 'told me to'
    ]):
        quoted.append(text)

print(f"Posts containing quoted messages: {len(quoted)}")
print(f"\nSamples:")
for t in quoted[:5]:
    print(f"\n{t[:400]}")

Posts containing quoted messages: 43

Samples:

I Scammed the scammers This is how I Scammed some scammers out of $650 by exploiting the clear weakness in their pig butchering or romance scam (NOT recommending anybody to play this game, I was just dumb to try and got lucky)

Chick from Hong Kong (Mary Mine presumably) texted my nr. "by accident", we hit it off to a point it started to feel romantic. (She even randomly video called me, they actu
--------------------------------------------------

Crypto Recovery Guru Scam \*PLEASE READ\*

CRYPTO RECOVERY SCAM

There are multiple users on this subreddit offering people "a way to get their crypto back" by referring them to a "Recovery Expert." This is a SCAM. NO ONE can hack your crypto back, regardless of what you are told. They will take whatever you give them and run away. It is 100% your responsibility. If you got scammed once, please ta
--------------------------------------------------

MEV Bots If someone offers you their code for 